## Importing Libraries

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt

## Getting the Data

In [0]:
airports_file = '/Volumes/workspace/flights/bronze/flights/airports.csv'

file_type = 'csv'

infer_schema = 'true'
first_row_is_header = 'true'
delimiter = ','

## Exploration (EDA)

In [0]:
airports = spark.read.format(file_type)\
    .option('inferSchema', infer_schema)\
        .option('header', first_row_is_header)\
            .option('sep', delimiter)\
                .load(airports_file)

In [0]:
display(airports.limit(5))

IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.4404
ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.6819
ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447


In [0]:
print(airports.count(), len(airports.columns))

322 7


In [0]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [0]:
# Checking null values
null_counts = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in airports.columns]

airports.select(null_counts).display()

IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,0,0,0,0,3,3


In [0]:
airports.where(airports.LATITUDE.isNull()).display()

IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
ECP,Northwest Florida Beaches International Airport,Panama City,FL,USA,null,null
PBG,Plattsburgh International Airport,Plattsburgh,NY,USA,null,null
UST,Northeast Florida Regional Airport (St. Augustine Airport),St. Augustine,FL,USA,null,null


In [0]:
airports.select("COUNTRY").distinct().show()

+-------+
|COUNTRY|
+-------+
|    USA|
+-------+



- I don't think I'll use LATITUDE AND LONGITUDE columns
- Will remove COUNTRY as well

In [0]:
airports = airports.drop("COUNTRY", "LATITUDE", "LONGITUDE")

## Saving to Silver

In [0]:
to_silver = '/Volumes/workspace/flights/silver/airports/'

airports.coalesce(1).write.csv(to_silver, mode='overwrite', header=True)